# Dual-Asset (30% Sell) Portfolio Bot — BTC + BNB Progressive Reinvestment

**Strategy:**
- Inject $20/month into shared USDT reserve
- Every red monthly candle: buy $10 BTC + reinvest% AND $10 BNB + reinvest%
- Sell 30% of each at their own new ATH close
- BTC params: 1%+2%->50% | BNB params: 1%+1%->40%

In [ ]:
import pandas as pd, requests, time, numpy as np, matplotlib.pyplot as plt, matplotlib.dates as mdates
from datetime import datetime, timezone
pd.set_option('display.max_columns', None); pd.set_option('display.width', 250)
pd.set_option('display.float_format', lambda x: '%.6f' % x)

In [ ]:
def fetch_klines(symbol):
    url = 'https://api.binance.com/api/v3/klines'
    all_k = []; st = 0
    while True:
        p = {'symbol': symbol, 'interval': '1M', 'startTime': st, 'limit': 500}
        d = requests.get(url, params=p).json()
        if not d: break
        all_k.extend(d)
        if len(d) < 500: break
        st = d[-1][0] + 1; time.sleep(0.1)
    return all_k

cols = ['timestamp','open','high','low','close','volume','close_time','quote_vol','trades','taker_buy_base','taker_buy_quote','ignore']

btc_k = fetch_klines('BTCUSDT')
bnb_k = fetch_klines('BNBUSDT')

btc = pd.DataFrame(btc_k, columns=cols); btc['date'] = pd.to_datetime(btc['timestamp'], unit='ms', utc=True)
bnb = pd.DataFrame(bnb_k, columns=cols); bnb['date'] = pd.to_datetime(bnb['timestamp'], unit='ms', utc=True)
for c in ['open','high','low','close','volume']:
    btc[c] = btc[c].astype(float); bnb[c] = bnb[c].astype(float)
btc = btc[['date','open','high','low','close','volume']].copy().sort_values('date').reset_index(drop=True)
bnb = bnb[['date','open','high','low','close','volume']].copy().sort_values('date').reset_index(drop=True)

# Align to common date range
start = max(btc['date'].min(), bnb['date'].min())
end = min(btc['date'].max(), bnb['date'].max())
btc = btc[(btc['date'] >= start) & (btc['date'] <= end)].reset_index(drop=True)
bnb = bnb[(bnb['date'] >= start) & (bnb['date'] <= end)].reset_index(drop=True)
print(f'BTC: {btc["date"].min().strftime("%Y-%m")} → {btc["date"].max().strftime("%Y-%m")} ({len(btc)} months)')
print(f'BNB: {bnb["date"].min().strftime("%Y-%m")} → {bnb["date"].max().strftime("%Y-%m")} ({len(bnb)} months)')

In [ ]:
# Run dual-asset portfolio backtest
# BTC: start=1%, +2%/buy, cap=50%
# BNB: start=1%, +1%/buy, cap=40%

usdt_reserve = 0.0
total_external_injected = 0.0

# BTC state
btc_held = 0.0; btc_highest = 0.0; btc_reinvest = 1.0; btc_invested_base = 0.0
# BNB state
bnb_held = 0.0; bnb_highest = 0.0; bnb_reinvest = 1.0; bnb_invested_base = 0.0

records = []

# Merge on month index (both have same length & aligned dates)
for i in range(len(btc)):
    btc_row = btc.iloc[i]; bnb_row = bnb.iloc[i]
    date = btc_row['date']
    btc_close = btc_row['close']; bnb_close = bnb_row['close']
    action_log = []

    # 1. External injection: $20 into reserve
    usdt_reserve += 20.0
    total_external_injected += 20.0

    # 2. BTC buy on red candle
    if btc_close < btc_row['open']:
        extra_btc = usdt_reserve * (btc_reinvest / 100.0) if usdt_reserve > 0 else 0.0
        btc_spend = min(10.0 + extra_btc, usdt_reserve)  # safety
        btc_bought = btc_spend / btc_close
        btc_held += btc_bought
        btc_invested_base += 10.0
        usdt_reserve -= btc_spend
        action_log.append(f'BTC buy ${btc_spend:.1f} @ {btc_close:.0f}')

    # 3. BNB buy on red candle
    if bnb_close < bnb_row['open']:
        extra_bnb = usdt_reserve * (bnb_reinvest / 100.0) if usdt_reserve > 0 else 0.0
        bnb_spend = min(10.0 + extra_bnb, usdt_reserve)
        bnb_bought = bnb_spend / bnb_close
        bnb_held += bnb_bought
        bnb_invested_base += 10.0
        usdt_reserve -= bnb_spend
        action_log.append(f'BNB buy ${bnb_spend:.1f} @ {bnb_close:.0f}')

    # 4. BTC sell at new ATH
    btc_prev = btc_highest
    if btc_close > btc_highest: btc_highest = btc_close
    if btc_held > 0.000001 and btc_close > btc_prev:
        sell = btc_held * 0.3
        usdt_reserve += sell * btc_close
        btc_held -= sell
        btc_reinvest = 1.0
        action_log.append(f'BTC sell 30% @ {btc_close:.0f} (-{sell:.6f})')
    else:
        if btc_reinvest < 50:
            btc_reinvest = min(50, btc_reinvest + 2)

    # 5. BNB sell at new ATH
    bnb_prev = bnb_highest
    if bnb_close > bnb_highest: bnb_highest = bnb_close
    if bnb_held > 0.000001 and bnb_close > bnb_prev:
        sell = bnb_held * 0.3
        usdt_reserve += sell * bnb_close
        bnb_held -= sell
        bnb_reinvest = 1.0
        action_log.append(f'BNB sell 30% @ {bnb_close:.0f} (-{sell:.6f})')
    else:
        if bnb_reinvest < 40:
            bnb_reinvest = min(40, bnb_reinvest + 1)

    btc_value = btc_held * btc_close
    bnb_value = bnb_held * bnb_close
    portfolio = btc_value + bnb_value + usdt_reserve

    records.append({
        'date': date, 'btc_close': btc_close, 'bnb_close': bnb_close,
        'btc_held': btc_held, 'btc_value': btc_value, 'btc_reinvest': btc_reinvest,
        'bnb_held': bnb_held, 'bnb_value': bnb_value, 'bnb_reinvest': bnb_reinvest,
        'usdt_reserve': usdt_reserve, 'total_injected': total_external_injected,
        'portfolio': portfolio, 'action': ' | '.join(action_log) if action_log else 'nothing',
        'btc_invested': btc_invested_base, 'bnb_invested': bnb_invested_base,
    })

res = pd.DataFrame(records)
print(f'Total months: {len(res)}')
print(f'Months with action: {len(res[res["action"] != "nothing"])}')
buy_months = len(res[res['action'].str.contains('buy', na=False)])
sell_months = len(res[res['action'].str.contains('sell', na=False)])
print(f'Buy months: {buy_months} | Sell months: {sell_months}')

In [ ]:
# Compute all metrics

final = res.iloc[-1]
total_invested = final['total_injected']
portfolio_value = final['portfolio']
total_pnl = portfolio_value - total_invested
ret_pct = (portfolio_value / total_invested - 1) * 100

eq = res['portfolio']
running_max = eq.cummax()
dd_raw = (running_max - eq) / running_max
max_dd = dd_raw.max() * 100

m = eq.pct_change().dropna()
m = m[m.index >= 12]
sharpe = (m.mean() / m.std()) * np.sqrt(12) if m.std() > 0 else 0
pos = m[m > 0].sum(); neg = m[m < 0].abs().sum()
pf = pos / neg if neg > 0 else float('inf')
ann = (1 + ret_pct/100) ** (12/len(res)) - 1
calmar = ann / (max_dd/100) if max_dd > 0 else 0

# BTC standalone metrics
btc_eq = res['btc_value'] + res['usdt_reserve']  # simplified: btc + shared reserve
# Actually compute BTC-only return
btc_final = final['btc_value'] + final['usdt_reserve'] * (final['btc_value'] / (final['btc_value'] + final['bnb_value'] + 1))  # rough
btc_ret = (final['btc_value'] / final['btc_invested'] - 1) * 100 if final['btc_invested'] > 0 else 0
bnb_ret = (final['bnb_value'] / final['bnb_invested'] - 1) * 100 if final['bnb_invested'] > 0 else 0
combined_ret = (portfolio_value / total_invested - 1) * 100

print('='*70)
print('PORTFOLIO BOT — DUAL ASSET RESULTS')
print('='*70)
print(f'Period:                {res["date"].min().strftime("%Y-%m")} → {res["date"].max().strftime("%Y-%m")} ({len(res)} months)')
print(f'External injection:    $20/month = ${total_invested:.0f} total')
print()
print('--- ASSET BREAKDOWN ---')
print(f'BTC invested base:     ${final["btc_invested"]:.0f}  (${final["btc_invested"]/len(res):.2f}/mo)')
print(f'BTC held:              {final["btc_held"]:.8f}  (${final["btc_value"]:.2f} @ {final["btc_close"]:.0f})')
print(f'BTC return (base only): {btc_ret:.1f}%')
print(f'BNB invested base:     ${final["bnb_invested"]:.0f}  (${final["bnb_invested"]/len(res):.2f}/mo)')
print(f'BNB held:              {final["bnb_held"]:.8f}  (${final["bnb_value"]:.2f} @ {final["bnb_close"]:.0f})')
print(f'BNB return (base only): {bnb_ret:.1f}%')
print()
print('--- PORTFOLIO ---')
print(f'USDT reserve:          ${final["usdt_reserve"]:.2f}')
print(f'Portfolio value:       ${portfolio_value:.2f}')
print(f'Total P&L:             ${total_pnl:.2f}')
print(f'Return:                {ret_pct:.2f}%')
print(f'Max drawdown:          {max_dd:.2f}%')
print(f'Sharpe ratio:          {sharpe:.2f}')
print(f'Profit factor:         {pf:.2f}')
print(f'Calmar ratio:          {calmar:.2f}')
print()
print('--- CASHFLOW ---')
print(f'Total injected:        ${total_invested:.0f}')
print(f'In BTC base:           ${final["btc_invested"]:.0f}')
print(f'In BNB base:           ${final["bnb_invested"]:.0f}')
print(f'Reinvested from reserve: ${total_invested - final["btc_invested"] - final["bnb_invested"]:.0f}')
print(f'Reserve remaining:     ${final["usdt_reserve"]:.2f}')

In [ ]:
# Chart 1: Portfolio dashboard

fig = plt.figure(figsize=(16, 14))
gs = fig.add_gridspec(5, 2, height_ratios=[3, 1, 1.2, 1.2, 1.2])

# --- Main: Portfolio value + invested ---
ax = fig.add_subplot(gs[0, :])
ax.fill_between(res['date'], res['total_injected'], res['portfolio'],
                where=res['portfolio'] >= res['total_injected'], color='green', alpha=0.12, label='Profit')
ax.fill_between(res['date'], res['portfolio'], res['total_injected'],
                where=res['portfolio'] < res['total_injected'], color='red', alpha=0.12, label='Loss')
ax.plot(res['date'], res['total_injected'], color='gray', linewidth=1.5, linestyle='--', label='Total Injected ($20/mo)')
ax.plot(res['date'], res['portfolio'], color='#2563eb', linewidth=2, label='Portfolio Value')

buys = res[res['action'].str.contains('buy', na=False)]
sells = res[res['action'].str.contains('sell', na=False)]
ax.scatter(buys['date'], buys['portfolio'], color='#16a34a', s=20, marker='^', zorder=5, alpha=0.6, label='Buy')
ax.scatter(sells['date'], sells['portfolio'], color='#dc2626', s=40, marker='v', zorder=5, label='Sell 30%')

ax.set_title('Dual-Asset Portfolio Bot — BTC + BNB (30% Sell) 30% Sell at ATH ($20/mo injection)', fontsize=13, fontweight='bold')
ax.set_ylabel('USDT'); ax.legend(loc='upper left', fontsize=8, ncol=3); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# --- Drawdown ---
ax = fig.add_subplot(gs[1, :])
ax.fill_between(res['date'], 0, dd_raw * 100, color='#dc2626', alpha=0.35)
ax.plot(res['date'], dd_raw * 100, color='#dc2626', linewidth=0.8)
ax.axhline(y=max_dd, color='#991b1b', linestyle=':', linewidth=1, label=f'Max DD: {max_dd:.1f}%')
ax.set_ylabel('Drawdown (%)'); ax.set_ylim(bottom=0); ax.legend(loc='lower left', fontsize=9); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# --- BTC value + reinvest rate ---
ax = fig.add_subplot(gs[2, 0])
ax.fill_between(res['date'], 0, res['btc_value'], color='#f59e0b', alpha=0.3)
ax.plot(res['date'], res['btc_value'], color='#d97706', linewidth=1.5)
ax.set_ylabel('BTC Value (USDT)'); ax.set_title('BTC Position'); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax = fig.add_subplot(gs[2, 1])
ax.step(res['date'], res['btc_reinvest'], where='post', color='#f59e0b', linewidth=1.5)
ax.axhline(y=50, color='#f59e0b', linestyle=':', alpha=0.5, label='Cap: 50%')
ax.set_ylabel('BTC Reinvest %'); ax.set_ylim(0, 58); ax.legend(fontsize=8); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# --- BNB value + reinvest rate ---
ax = fig.add_subplot(gs[3, 0])
ax.fill_between(res['date'], 0, res['bnb_value'], color='#8b5cf6', alpha=0.3)
ax.plot(res['date'], res['bnb_value'], color='#7c3aed', linewidth=1.5)
ax.set_ylabel('BNB Value (USDT)'); ax.set_title('BNB Position'); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax = fig.add_subplot(gs[3, 1])
ax.step(res['date'], res['bnb_reinvest'], where='post', color='#8b5cf6', linewidth=1.5)
ax.axhline(y=40, color='#8b5cf6', linestyle=':', alpha=0.5, label='Cap: 40%')
ax.set_ylabel('BNB Reinvest %'); ax.set_ylim(0, 46); ax.legend(fontsize=8); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# --- USDT reserve + prices ---
ax = fig.add_subplot(gs[4, 0])
ax.fill_between(res['date'], 0, res['usdt_reserve'], color='#10b981', alpha=0.3)
ax.plot(res['date'], res['usdt_reserve'], color='#059669', linewidth=1.5)
ax.set_ylabel('USDT Reserve'); ax.set_title('Cash Reserve'); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax = fig.add_subplot(gs[4, 1])
ax.plot(res['date'], res['btc_close'], color='#f59e0b', linewidth=1, alpha=0.7, label='BTC')
axb = ax.twinx()
axb.plot(res['date'], res['bnb_close'], color='#8b5cf6', linewidth=1, alpha=0.7, label='BNB')
ax.set_ylabel('BTC (USDT)'); axb.set_ylabel('BNB (USDT)')
ax.set_title('Prices'); ax.grid(True, alpha=0.25)
ax.legend(loc='upper left', fontsize=8); axb.legend(loc='upper right', fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig('dca-bt-buy-bot-portfolio-sell30.png', dpi=150, bbox_inches='tight')
print('Chart saved as dca-bt-buy-bot-portfolio-sell30.png')
plt.show()

In [ ]:
# Chart 2: Stacked asset allocation over time

fig, ax = plt.subplots(figsize=(14, 7))

ax.stackplot(res['date'], res['btc_value'], res['bnb_value'], res['usdt_reserve'],
             labels=['BTC', 'BNB', 'USDT Reserve'],
             colors=['#f59e0b', '#8b5cf6', '#10b981'], alpha=0.8)
ax.plot(res['date'], res['total_injected'], color='gray', linewidth=1.5, linestyle='--', label='Total Injected')
ax.set_title('Portfolio Composition Over Time', fontsize=13, fontweight='bold')
ax.set_ylabel('USDT'); ax.legend(loc='upper left', fontsize=9); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig('dca-bt-buy-bot-portfolio-sell30-stacked.png', dpi=150, bbox_inches='tight')
print('Chart saved as dca-bt-buy-bot-portfolio-sell30-stacked.png')
plt.show()

In [ ]:
# Monthly cashflow summary

cashflow = []
for i, row in res.iterrows():
    injection = 20.0
    btc_buy = 0; bnb_buy = 0; btc_sell = 0; bnb_sell = 0
    # Reconstruct buys/sells from action string
    if 'BTC buy' in row['action']:
        import re
        m = re.search(r'BTC buy \$(\d+\.?\d*)', row['action'])
        if m: btc_buy = float(m.group(1))
    if 'BNB buy' in row['action']:
        m = re.search(r'BNB buy \$(\d+\.?\d*)', row['action'])
        if m: bnb_buy = float(m.group(1))
    # For sells we can compute from the change in holdings (simpler)
    if i > 0:
        btc_delta = row['btc_held'] - res.iloc[i-1]['btc_held']
        if btc_delta < -0.000001:
            btc_sell = abs(btc_delta) * (row['btc_close'] + res.iloc[i-1]['btc_close']) / 2
        bnb_delta = row['bnb_held'] - res.iloc[i-1]['bnb_held']
        if bnb_delta < -0.000001:
            bnb_sell = abs(bnb_delta) * (row['bnb_close'] + res.iloc[i-1]['bnb_close']) / 2
    
    reserve_delta = row['usdt_reserve'] - (res.iloc[i-1]['usdt_reserve'] if i > 0 else 0)
    cashflow.append({
        'date': row['date'], 'injection': injection, 'btc_buy': btc_buy, 'bnb_buy': bnb_buy,
        'btc_sell': btc_sell, 'bnb_sell': bnb_sell,
        'reserve_delta': reserve_delta, 'reserve': row['usdt_reserve'],
        'portfolio': row['portfolio']
    })

cf = pd.DataFrame(cashflow)
print(f"{'Date':>10s} {'Inj':>6s} {'BTCbuy':>8s} {'BNBbuy':>8s} {'BTCsell':>8s} {'BNBsell':>8s} {'Resv':>8s} {'Portfolio':>10s}")
print('-'*74)
for _, r in cf.iterrows():
    print(f"{r['date'].strftime('%Y-%m'):>10s} {r['injection']:>6.0f} {r['btc_buy']:>8.1f} {r['bnb_buy']:>8.1f} {r['btc_sell']:>8.1f} {r['bnb_sell']:>8.1f} {r['reserve']:>8.1f} {r['portfolio']:>10.1f}")

print()
print(f"Total injection: ${cf['injection'].sum():.0f}")
print(f"Total BTC buys:  ${cf['btc_buy'].sum():.0f}")
print(f"Total BNB buys:  ${cf['bnb_buy'].sum():.0f}")
print(f"Total BTC sells: ${cf['btc_sell'].sum():.0f}")
print(f"Total BNB sells: ${cf['bnb_sell'].sum():.0f}")
print(f"Final reserve:   ${cf['reserve'].iloc[-1]:.0f}")
print(f"Final portfolio: ${cf['portfolio'].iloc[-1]:.0f}")